# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. The dataset describes mass spectrometry analysis of proteins from extracellular vesicles (EVs) in human (Homo sapiens) samples, characterizing protein abundance, modifications, and features.

### Dataset Source
The dataset source is provided as a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields (columns), and their unique `@id` values. We will use `mlcroissant` to list all record sets and, for each, display their fields/columns. All references are with `@id` as required.

In [ ]:
print("Available Record Sets (by @id and name):")
record_sets = list(metadata.record_sets)
for rec in record_sets:
    print(f"  @id: {rec.id}  |  name: {rec.name}")
    print("    Fields/Columns:")
    for fld in rec.fields:
        print(f"      @id: {fld.id}  |  name: {fld.name}  |  dataType: {fld.data_type}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Specify the `@id` string of each record set. We'll show columns for the first record set as an example.

In [ ]:
# Collect list of record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load all data into DataFrames
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for Record Set @id: {rs_id}")

# Display fields/columns for the first record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for record set @id: {main_record_set_id}\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main dataset:
- Filtering for rows with a high number of unique peptides
- Normalizing a protein abundance field
- Grouping and aggregating by a category

*All field names are referenced by their `@id` as listed in the overview above.*

In [ ]:
# Choose a record set and fields for EDA
# You may need to update these @id values based on the actual output from section 2.
record_set_id = main_record_set_id  # Use primary record set @id

# Example: Find field ids pertaining to unique peptides and abundance (update accordingly)
fields = {fld.name: fld.id for fld in [f for f in metadata.record_sets[0].fields]}
# Print available fields for user mapping
print(fields)

# Supply field @ids -- adjust as appropriate for your dataset
numeric_field = None
for name in fields:
    if "unique_peptide" in name.lower() or "unique" in name.lower():
        numeric_field = fields[name]
        break
if numeric_field is None:
    # Fallback to first numeric column
    for col in dataframes[record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[record_set_id][col]):
            numeric_field = col
            break

threshold = 2  # Example threshold for filtering
filtered_df = dataframes[record_set_id].copy()

if numeric_field is not None and numeric_field in filtered_df.columns:
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold} (by @id): {len(filtered_df)} records")
    
    # Normalizing numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a field/category (e.g., protein name or category if available)
    group_field = None
    for name in fields:
        if "category" in name.lower() or "type" in name.lower() or "group" in name.lower():
            group_field = fields[name]
            break
    if group_field is None and len(filtered_df.columns) > 1:
        # Fallback to second column
        group_field = filtered_df.columns[1]
    
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for filtering and normalization.")

## 5. Visualization
Visualize distributions of numeric and category fields in the main record set using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot a histogram for the numeric field
if numeric_field and numeric_field in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Plot mean value per group if grouping is possible
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} by {group_field} (by @id)")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha="right")
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded Croissant metadata and records using `mlcroissant`
- Explored available record sets and fields via their `@id`
- Extracted and loaded table data into Pandas DataFrames
- Applied simple EDA including filtering, normalization, and grouping by field `@id`
- Visualized data distributions and group means

This workflow establishes a reproducible approach to accessing, processing, and analyzing Croissant-encoded mass-spec datasets using the correct entity references. Update field `@id` values as needed depending on your dataset version for more granular analysis.